In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\lenovo\anaconda3\envs\mle-housing\python.exe
3.0.4
c:\Users\lenovo\anaconda3\envs\mle-housing\Lib\site-packages\xgboost\__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\lenovo\anaconda3\envs\mle-housing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==============================================
# 2. Load processed datasets
# ==============================================
df_train = pd.read_csv('../data/processed/feature_engineered_train.csv')
df_eval  = pd.read_csv('../data/processed/feature_engineered_eval.csv')


# Define target + features
target = 'price'
X_train, y_train = df_train.drop(columns=[target]), df_train[target]
X_eval, y_eval   = df_eval.drop(columns=[target]), df_eval[target]

print('Train shape:', X_train.shape)
print('Eval shape:', X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({'rmse': rmse, 'mae': mae, 'r2': r2})

    return rmse

In [6]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri('/Users/lenovo/Arif/Project/house-price-prediction/mlruns')
mlflow.set_experiment('xgboost_optuna_housing')

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15)

print('Best params:', study.best_trial.params)

c:\Users\lenovo\anaconda3\envs\mle-housing\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/02/19 15:09:24 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-02-19 15:09:27,393] A new study created in memory with name: no-name-d09e3d4c-793c-4365-bc1c-32adb89dd11b
[I 2026-02-19 15:10:09,444] Trial 0 finished with value: 81717.96979507692 and parameters: {'n_estimators': 306, 'max_depth': 7, 'learning_rate': 0.14110102624279733, 'subsample': 0.776433522

Best params: {'n_estimators': 943, 'max_depth': 10, 'learning_rate': 0.025467612400597864, 'subsample': 0.5603715804119231, 'colsample_bytree': 0.5053012779064261, 'min_child_weight': 10, 'gamma': 1.7049995570266179, 'reg_alpha': 0.00018699841416400844, 'reg_lambda': 4.906471027583702}


In [ ]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print('Final tuned model performance:')
print('MAE:', mae)
print('RMSE:', rmse)
print('R²:', r2)

# Log final model
with mlflow.start_run(run_name='best_xgboost_model'):
    mlflow.log_params(best_params)
    mlflow.log_metrics({'rmse': rmse, 'mae': mae, 'r2': r2})
    mlflow.sklearn.log_model(sk_model = best_model, name='model')
    print('✅ Model successfully logged to MLflow!')

Final tuned model performance:
MAE: 30274.737702293445
RMSE: 67817.98983832134
R²: 0.9644573457393042


c:\Users\lenovo\anaconda3\envs\mle-housing\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


✅ Model berhasil disimpan ke MLflow!
